# Vision Transformers, Self-Supervised Learning and VLMs


Lectures 10 to 15 built the transformer for text. This lecture brings it to
images and asks the question that text never had to face: where do the labels
come from? Text pretraining is self-supervised by construction (the next token
is the label). Images arrive with nothing attached, so the field had to invent
objectives that make an encoder learn from pixels alone. The lecture has four
parts. First the Vision Transformer, which turns an image into a sequence of
patch tokens so that the machinery of lecture 10 applies unchanged, and CLIP,
which pairs images with captions and learns both encoders contrastively. Then
the core of the lecture: four families of self-supervised objectives
(contrastive, masked, self-distillation and joint-embedding prediction), what
each one rewards, and why DINO features became the default backbone. Then the
foundation models built on those backbones (SAM, Grounding DINO) and, finally,
the vision-language model pattern that every current multimodal LLM uses: a
vision encoder, a projector, and a language model, with a token budget you can
compute by hand.

Notation. UDL covers transformers (chapter 12) but not self-supervised vision,
so from the SimCLR slide onward the lecture uses each primary paper's notation
and says so: $\mathbf{z}$ for projected embeddings and $\tau$ for temperature
(SimCLR), $P_s$ and $P_t$ for student and teacher distributions (DINO). The
quizzable items are the patch-count arithmetic, the gradient of the NT-Xent
loss, and the visual-token and KV-cache calculation at the end.


## Today's roadmap

1. From pixels to patches: the Vision Transformer (ViT)
2. CLIP: learning an image encoder from captions
3. Self-supervised pretraining for vision: contrastive, masked, self-distillation, JEPA
4. Foundation models for vision: SAM, SAM 2, Grounding DINO; fine-tune or freeze?
5. Vision-language models: encoder + projector + LLM, and the visual-token budget
6. Summary


# From pixels to patches: the Vision Transformer

Lecture 7 ended with convolutional networks as the architecture for images, and
lecture 10 introduced self-attention for sequences. This section shows that a
transformer encoder needs almost no change to classify images, once an image is
cut into patches, and that the price of dropping the convolutional inductive
bias is paid in data.


## Why transformers for vision?

![ImageNet top-1 error of the state of the art by year. Based on Papers with Code.](figs/16-Vision-Transformers-and-SSL/imagenet-top1-error-history.png)


## The Vision Transformer (ViT)

- [Dosovitskiy *et al.*, "An Image is Worth 16x16 Words"](https://arxiv.org/abs/2010.11929) (arXiv Oct. 2020, ICLR 2021)
- Cut the image into $P \times P$ patches; each patch is one token. A $224 \times 224$ image at $P = 16$ is a sequence of 196 tokens, which is the length of a paragraph, not a pixel grid
- Feed the sequence to a standard transformer *encoder* (lecture 10): no convolutions, no 2-D structure except a learned position embedding
- Performs poorly when trained on ImageNet-1K alone: a transformer lacks the locality and translation-equivariance bias of a CNN
- Matches or beats the best CNNs when pre-trained on 14M to 300M labeled images

**Note**

Large-scale training trumps inductive bias.


## ViT: putting it all together


1. Divide image into $P \times P$ patches
2. Create sequence of length $N = HW/P^2$
3. Flatten the patches and map to $D$ dimensions with a trainable linear projection
4. Add a learned 1-D position embedding
5. Prepend a learnable `[CLS]` embedding (as in BERT, 2018)
6. Transformer encoder, then an MLP head on the `[CLS]` output

![Vision Transformer (ViT). Figure 1 of [Dosovitskiy et al., 2020](https://arxiv.org/abs/2010.11929).](figs/16-Vision-Transformers-and-SSL/vit-architecture.png)


## The patch arithmetic (quizzable)

For an $H \times W \times 3$ image and patch size $P$:

$$
N = \frac{HW}{P^2} \quad \text{tokens}, \qquad
\text{each a vector of } 3P^2 \text{ numbers}, \qquad
\mathbf{E} \in \mathbb{R}^{3P^2 \times D}.
$$

- $224 \times 224$, $P = 16$: $N = 14 \times 14 = 196$ tokens, each $16 \cdot 16 \cdot 3 = 768$ numbers. Sequence length into the encoder: $N + 1 = 197$ with `[CLS]`.
- The patch embedding $\mathbf{E}$ for ViT-Base ($D = 768$) has $768 \times 768 \approx 590\text{K}$ parameters: a tiny fraction of the 86M.
- Self-attention costs $\mathcal{O}(N^2 D)$ per layer (lecture 10). Halving $P$ to 8 gives $N = 784$ and **16 times** the attention cost. Doubling resolution to $448$ at $P = 16$ gives $N = 784$ as well.
- Fine-tuning at higher resolution keeps $P$, so $N$ grows; the position embeddings are 2-D interpolated to the new grid.



The ViT's first step in code: cut a (random) image into $P \times P$ patches,
flatten each one, and project it to $D$ dimensions with $\mathbf{E}$. Two
reshapes and a transpose do the cutting. Change `H`, `W`, `P` or `D` and rerun;
the table at the end gives the token count and the relative attention cost for
the cases on the slide and in the notes.

In [ ]:
import numpy as np

H, W, P, D = 224, 224, 16, 768     # image size, patch size, embedding width (ViT-B/16)
rng = np.random.default_rng(0)
image = rng.random((H, W, 3))

def patchify(img, P):
    """(H, W, C) -> (N, P*P*C): row-major grid of flattened patches."""
    H, W, C = img.shape
    grid = img.reshape(H // P, P, W // P, P, C).transpose(0, 2, 1, 3, 4)
    return grid.reshape(-1, P * P * C)

patches = patchify(image, P)
assert np.array_equal(patches[1], image[:P, P:2 * P].ravel())   # patch (row 0, col 1)

E = rng.normal(0, 0.02, (3 * P * P, D))   # the trainable patch embedding
tokens = patches @ E
print(f"image {H}x{W}x3 -> patches {patches.shape} -> tokens {tokens.shape}")
print(f"N = HW/P^2 = {H * W // P**2}; sequence with [CLS]: {H * W // P**2 + 1}")
print(f"E has 3P^2 x D = {3 * P * P} x {D} = {E.size:,} weights "
      f"({E.size + D:,} with a bias); ViT-B has about 86M")

base = (224 * 224 // 16**2) ** 2
print(f"\n{'image':>6} {'P':>3} {'N':>5} {'attention cost vs 224/16':>25}")
for side, p in [(224, 16), (224, 8), (448, 16), (384, 16), (336, 14)]:
    n = side * side // p**2
    print(f"{side:>6} {p:>3} {n:>5} {n**2 / base:>24.1f}x")

## ViT encoder layer

![](figs/16-Vision-Transformers-and-SSL/vit-transformer-encoder-layer.png)


- The pre-norm transformer block of lecture 10: LayerNorm *before* attention and before the MLP, inside the residual path
- Identical to the block in GPT-2 and in the NLP transformers of the time, which is the point: the same scalable implementations and training recipes apply
- Only the input side is image-specific: the patch projection and the position embedding


## ViT training datasets and model variants

| Dataset | # Classes | # Images |
|---|---|---|
| ILSVRC-2012 (ImageNet-1K) | 1K | 1.3M |
| ImageNet-21K | 21K | 14M |
| JFT-300M (Google-internal) | 18K | 303M |

| Model | Layers | Hidden size $D$ | MLP size | Heads | Params | |
|---|---|---|---|---|---|---|
| ViT-Base | 12 | 768 | 3072 | 12 | 86M | same as BERT-Base |
| ViT-Large | 24 | 1024 | 4096 | 16 | 307M | same as BERT-Large |
| ViT-Huge | 32 | 1280 | 5120 | 16 | 632M | new for ViT |

Notation: **ViT-L/16** is the Large variant with **$16 \times 16$ input patches** (the number after the slash is the patch size $P$, not the image size). Smaller $P$ means more tokens and more compute.

## The data-scale result

| Pre-trained on | JFT-300M (ViT-H/14) | JFT-300M (ViT-L/16) | ImageNet-21K (ViT-L/16) | JFT-300M BiT-L (ResNet152x4) | Noisy Student (EfficientNet-L2) |
|---|---|---|---|---|---|
| ImageNet | **88.55** ± 0.04 | 87.76 ± 0.03 | 85.30 ± 0.02 | 87.54 ± 0.02 | 88.4/88.5\* |
| ImageNet ReaL | **90.72** ± 0.05 | 90.54 ± 0.03 | 88.62 ± 0.05 | 90.54 | 90.55 |
| CIFAR-100 | **94.55** ± 0.04 | 93.90 ± 0.05 | 93.25 ± 0.05 | 93.51 ± 0.08 | – |
| VTAB (19 tasks) | **77.63** ± 0.23 | 76.28 ± 0.46 | 72.72 ± 0.21 | 76.29 ± 1.70 | – |
| TPUv3-core-days | 2.5k | 0.68k | 0.23k | 9.9k | 12.3k |

Table 2 of [Dosovitskiy et al., 2020](https://arxiv.org/abs/2010.11929), abridged.

- The same ViT-L/16 gains 2.5 points on ImageNet going from 14M to 303M pre-training images; a ResNet of similar size does not
- ViT-L/16 pre-trained on JFT beats the ResNet at **15 times less compute**: the transformer is the more compute-efficient learner once there is enough data
- Follow-up: [Zhai et al., 2022](https://arxiv.org/abs/2106.04560) scale to ViT-G/14 with 2B parameters on JFT-3B and reach 90.45% top-1; with strong weight decay on the head, 84.86% from only 10 examples per class

## ViT: visualizing internals

![Figure 7 of [Dosovitskiy et al., 2020](https://arxiv.org/abs/2010.11929).](figs/16-Vision-Transformers-and-SSL/vit-visualizing-internals.png)

The learned patch embedding looks like the first layer of a CNN: oriented edges and color blobs.


The 1-D position embedding learns the 2-D grid on its own: each patch's embedding is most similar to its row and column neighbors.


Some heads attend locally from the first layer (like a small convolution); by the later layers every head attends globally.


# CLIP: learning an image encoder from captions

ViT's best results need hundreds of millions of *labeled* images, and the
labels are a fixed vocabulary. CLIP replaces labels with the captions that
already accompany images on the web, and trains the image encoder (a ViT) and a
text encoder together so that matching pairs agree. The objective it uses, a
contrastive loss, is also the first of the four self-supervised objectives in
the next section.


## CLIP (2021): contrastive language-image pretraining

- [Radford *et al.*, "Learning Transferable Visual Models From Natural Language Supervision"](https://arxiv.org/abs/2103.00020), ICML 2021
- Learn directly from raw text about images: a new 400M (image, text) pair dataset, WebImageText (WIT), scraped from the internet
- The pretraining task: given a batch of $n$ images and their $n$ captions, predict which caption goes with which image
- No fixed label set, so the learned encoders transfer **zero-shot** to many classification datasets: the class names become captions
- Shows promise for zero-shot OCR, action recognition, geo-localization, and much more

**Note**

Contrastive loss: maximize the similarity of matching (image, text) pairs and simultaneously minimize the similarity of the $n^2 - n$ non-matching pairs in the batch.


## CLIP overview

![Figure 1 of [Radford et al., 2021](https://arxiv.org/abs/2103.00020). (1) Contrastive pre-training on a batch of (image, text) pairs. (2) At test time the text encoder turns class names into a zero-shot linear classifier. (3) The image is assigned the caption it is most similar to.](figs/16-Vision-Transformers-and-SSL/clip-overview.png).' and encoded to T_1 to T_N. (3) Use for zero-shot prediction: a new image is encoded to I_1, compared with every T_j, and the best match, 'A photo of a dog.', is chosen." width="90%"}

## The two encoders

**Text encoder**

- Lower-cased byte-pair encoding, bracketed with `[SOS]` and `[EOS]`
- 12-layer, 512-wide transformer, 8 heads
- The `[EOS]` activation is the text feature


**Image encoder**

- Trained 5 ResNets and 3 ViTs: ViT-B/32, ViT-B/16, ViT-L/14
- Best: **ViT-L/14@336px**, a ViT-Large with $14 \times 14$ patches, fine-tuned one extra epoch at $336 \times 336$
- ViTs were about 3x more compute-efficient than the CLIP ResNets: RN50x64 took 18 days on 592 V100s, ViT-L/14 12 days on 256


Both outputs are projected by a learned matrix into one shared embedding space and L2-normalized, so a dot product is a cosine similarity.

## CLIP in twelve lines

```python
# image_encoder - ResNet or Vision Transformer
# text_encoder  - CBOW or Text Transformer
# I[n, h, w, c] - minibatch of aligned images
# T[n, l]       - minibatch of aligned texts
# W_i[d_i, d_e] - learned proj of image to embed
# W_t[d_t, d_e] - learned proj of text to embed
# t             - learned temperature parameter

# extract feature representations of each modality
I_f = image_encoder(I) #[n, d_i]
T_f = text_encoder(T)  #[n, d_t]

# joint multimodal embedding [n, d_e]
I_e = l2_normalize(np.dot(I_f, W_i), axis=1)
T_e = l2_normalize(np.dot(T_f, W_t), axis=1)

# scaled pairwise cosine similarities [n, n]
logits = np.dot(I_e, T_e.T) * np.exp(t)

# symmetric loss function
labels = np.arange(n)
loss_i = cross_entropy_loss(logits, labels, axis=0)
loss_t = cross_entropy_loss(logits, labels, axis=1)
loss   = (loss_i + loss_t)/2
```

Figure 3 of [Radford et al., 2021](https://arxiv.org/abs/2103.00020).

## Reading the pseudocode: similarities and temperature

**Cosine similarity.** For $\mathbf{a}, \mathbf{b} \in \mathbb{R}^{d_e}$,

$$
\cos\theta = \frac{\mathbf{a}^\top \mathbf{b}}{\|\mathbf{a}\|\,\|\mathbf{b}\|} \in [-1, 1],
$$

equal to $1$ for the same direction, $0$ when orthogonal, $-1$ for opposite directions. After `l2_normalize`, `np.dot(I_e, T_e.T)` is the $n \times n$ matrix of cosines: entry $(i, j)$ compares image $i$ with text $j$.


**Temperature.** The logits are the cosines times $\exp(t)$, with $t$ a learned scalar. $\exp(t)$ multiplies the logits, so it is an **inverse** temperature: in the usual softmax notation $\exp(\mathrm{logit}/\tau)$, $\tau = 1/\exp(t)$. Larger $t$ means a sharper softmax.

CLIP initializes $\tau = 0.07$ (so $\exp(t) \approx 14.3$) and clips $\exp(t)$ at 100, "which we found necessary to prevent training instability." Without the scale, cosines in $[-1, 1]$ give a nearly flat softmax over 32,768 candidates.


## The symmetric loss

With $s_{ij} = \exp(t)\, \mathbf{I}_i^\top \mathbf{T}_j$ the scaled cosine between image $i$ and text $j$, and the matching pairs on the diagonal ($\mathtt{labels} = [0, 1, \ldots, n-1]$):

$$
L_{\text{image}} = -\frac{1}{n}\sum_{i=1}^{n} \log \frac{\exp(s_{ii})}{\sum_{j=1}^{n} \exp(s_{ij})},
\qquad
L_{\text{text}} = -\frac{1}{n}\sum_{j=1}^{n} \log \frac{\exp(s_{jj})}{\sum_{i=1}^{n} \exp(s_{ij})},
$$

$$
L = \tfrac{1}{2}\left(L_{\text{image}} + L_{\text{text}}\right).
$$

- $L_{\text{image}}$: each image is a query and the softmax runs **over the texts** (along row $i$ of `logits`). That is `loss_i`.
- $L_{\text{text}}$: each text is a query and the softmax runs **over the images** (along column $j$). That is `loss_t`.
- The pseudocode's `axis=0` / `axis=1` are labels for "normalize per image" and "normalize per text", not NumPy's axis convention; implementations such as [OpenCLIP](https://github.com/mlfoundations/open_clip) compute `cross_entropy(logits, labels) + cross_entropy(logits.T, labels)`.
- Averaging the two makes the loss symmetric, so image-to-text and text-to-image retrieval are both optimized. Each direction is an $n$-way classification with $n - 1$ in-batch negatives, which is why CLIP uses a batch of 32,768.


The symmetric loss on a toy batch, following the pseudocode line by line. The
"encoders" are skipped: the image embeddings are random unit vectors, and the
text embeddings are noisy copies of them, so caption $i$ really does match
image $i$. Compare an aligned batch, the same batch with the captions shuffled,
and the aligned batch without the temperature scale. Chance level is
$\log n$.

In [ ]:
import numpy as np

n, d_e = 8, 64             # batch size, shared embedding width
tau = 0.07                 # CLIP's initial temperature, so exp(t) = 1/tau
rng = np.random.default_rng(0)

def l2_normalize(x):
    return x / np.linalg.norm(x, axis=1, keepdims=True)

def cross_entropy(logits, labels):
    """Mean over rows of -log softmax(row)[label]."""
    z = logits - logits.max(axis=1, keepdims=True)
    log_p = z - np.log(np.exp(z).sum(axis=1, keepdims=True))
    return -log_p[np.arange(len(labels)), labels].mean()

def clip_loss(I_e, T_e, scale):
    logits = I_e @ T_e.T * scale                 # [n, n] scaled cosines
    labels = np.arange(len(I_e))
    loss_i = cross_entropy(logits, labels)       # each image: softmax over texts
    loss_t = cross_entropy(logits.T, labels)     # each text: softmax over images
    return (loss_i + loss_t) / 2

I_e = l2_normalize(rng.normal(size=(n, d_e)))
T_e = l2_normalize(I_e + 0.5 * rng.normal(size=(n, d_e)) / np.sqrt(d_e))
shuffled = T_e[rng.permutation(n)]

print(f"exp(t) = 1/tau = {1 / tau:.1f}; chance level log n = {np.log(n):.3f}")
print(f"mean cosine: matching pairs {np.mean(np.sum(I_e * T_e, axis=1)):.2f}, "
      f"non-matching {np.mean((I_e @ T_e.T)[~np.eye(n, dtype=bool)]):.2f}")
print(f"aligned batch,  scale 1/tau: L = {clip_loss(I_e, T_e, 1 / tau):.4f}")
print(f"shuffled batch, scale 1/tau: L = {clip_loss(I_e, shuffled, 1 / tau):.4f}")
print(f"aligned batch,  scale 1:     L = {clip_loss(I_e, T_e, 1.0):.4f}")

Without the scale, even a perfectly matched batch stays far from zero loss:
the logits are cosines, at most 1 apart from the next candidate, and a softmax
over numbers that close is nearly flat. With a batch of 32,768 instead of 8
the loss could not get much below $\log n$. Scaling by $\exp(t)$ is what lets
the loss reward a match.


## Why a contrastive objective?

- First attempt: a transformer that *generates* the caption ([blue]{style="color: #1f77b4;"}). Slow: captions are free-form and most words are unpredictable
- Predicting a bag of words of the caption is 3x more data-efficient ([orange]{style="color: #ff7f0e;"})
- Only *matching* image to caption, the contrastive objective, is another 4x more efficient ([green]{style="color: #2ca02c;"})
- The lesson recurs in the next section: predict less, learn more


![Figure 2 of [Radford et al., 2021](https://arxiv.org/abs/2103.00020).](figs/16-Vision-Transformers-and-SSL/clip-contrastive-efficiency.png)


## Zero-shot classification


- Write each class name into a prompt template: "A photo of a {dog}."
- Encode the $K$ prompts once: $\mathbf{T}_1, \ldots, \mathbf{T}_K$. This *is* a linear classifier with weight matrix $[\mathbf{T}_1 \cdots \mathbf{T}_K]$, synthesized from text instead of learned from labeled examples
- Encode the image, take the softmax over the $K$ scaled cosines, predict the argmax
- No training data from the target dataset was used: zero-shot transfer. Prompt ensembling ("a photo of a big {}", "a photo of a small {}") adds a few points


## Zero-shot CLIP versus a supervised baseline

![Figure 5 of [Radford et al., 2021](https://arxiv.org/abs/2103.00020).](figs/16-Vision-Transformers-and-SSL/clip-zero-shot-vs-resnet50.png)


- Baseline: a ResNet-50 trained with ImageNet labels, frozen, with a linear classifier fitted on each of 27 datasets (a **linear probe**, defined properly in the next section)
- Zero-shot CLIP beats it on 16 of 27 datasets with no training examples at all
- On ImageNet, zero-shot ViT-L/14@336 reaches 76.2% top-1, the accuracy of the original supervised ResNet-50
- It loses badly where the web rarely captions what matters: satellite land use (EuroSAT), tumor histology (PatchCamelyon), distance to a car (KITTI). Captions are a biased supervisor



# Self-supervised pretraining for vision

CLIP still needs text. This section asks what an encoder can learn from
images alone, and follows the four objectives the field converged on between
2020 and 2025. Each one invents a prediction task whose labels come for free
from the image itself, and each rewards a different property of the learned
representation. The comparison slide at the end of the section is the one to
remember.


## Labels are the bottleneck; the pretext-task idea

- ImageNet took years and tens of thousands of crowd workers (ILSVRC reports ~49,000 Mechanical Turk workers) for 1.3M labels. JFT-300M is Google-internal. Text pretraining uses trillions of free tokens
- **Self-supervised learning (SSL)**: invent a *pretext task* whose labels are computed from the unlabeled image, train the encoder on it, keep the encoder and throw the task away
- Early pretext tasks (2016 to 2018): predict the rotation applied to an image ([Gidaris et al., 2018](https://arxiv.org/abs/1803.07728)), solve a jigsaw of shuffled patches ([Noroozi and Favaro, 2016](https://arxiv.org/abs/1603.09246)), colorize a grayscale image ([Zhang et al., 2016](https://arxiv.org/abs/1603.08511)), predict the next pixel autoregressively ([ImageGPT, Chen et al., 2020](https://openai.com/research/image-gpt))
- They all worked a little. The representations were far behind supervised ones, because solving the pretext task does not require understanding objects
- Four families since 2020 closed the gap and then passed supervised pretraining: **contrastive**, **masked**, **self-distillation**, and **joint-embedding prediction**



## Contrastive learning: SimCLR

![After Figure 2 of [Chen et al., 2020](https://arxiv.org/abs/2002.05709).](figs/16-Vision-Transformers-and-SSL/simclr-framework.png)


[Chen, Kornblith, Norouzi and Hinton, "A Simple Framework for Contrastive Learning of Visual Representations"](https://arxiv.org/abs/2002.05709), ICML 2020.

- Two random augmentations of the same image are a **positive pair**; the other $2(N-1)$ augmented images in the batch are the **negatives**
- Shared encoder $\mathrm{f}[\cdot]$ (a ResNet in the paper; a ViT in later work) gives $\mathbf{h}$; a small MLP head $\mathrm{g}[\cdot]$ gives $\mathbf{z}$
- Loss on $\mathbf{z}$; downstream tasks use $\mathbf{h}$
- CLIP is the same idea with the second view replaced by the caption


## The NT-Xent loss

For a positive pair $(i, j)$ among the $2N$ augmented examples, with $\mathrm{sim}(\mathbf{u}, \mathbf{v}) = \mathbf{u}^\top \mathbf{v} / \|\mathbf{u}\| \|\mathbf{v}\|$ and temperature $\tau$:

$$
\ell_{i,j} = -\log \frac{\exp\left(\mathrm{sim}(\mathbf{z}_i, \mathbf{z}_j)/\tau\right)}{\sum_{k=1}^{2N} \mathbb{1}[k \neq i]\, \exp\left(\mathrm{sim}(\mathbf{z}_i, \mathbf{z}_k)/\tau\right)}
$$

- A $(2N-1)$-way softmax classification: "which of the other examples is my partner?" (NT-Xent: normalized, temperature-scaled cross-entropy; the same form as InfoNCE, [van den Oord et al., 2018](https://arxiv.org/abs/1807.03748))
- The final loss sums $\ell_{i,j}$ and $\ell_{j,i}$ over all positive pairs in the batch
- CLIP's $L_{\text{image}}$ is this loss with the texts as the candidate set and $\exp(t) = 1/\tau$
- Default $\tau \approx 0.1$; batch sizes 256 to 8192 (bigger is better, because more negatives)


## Deriving the gradient of NT-Xent (quizzable)

Write $s_{ik} = \mathbf{z}_i^\top \mathbf{z}_k / \tau$ with every $\mathbf{z}$ already unit-normalized (treat the normalization as fixed), and $p_{ik} = \exp(s_{ik}) / \sum_{m \neq i} \exp(s_{im})$ for the softmax over candidates $k \neq i$. Then

$$
\ell_{i,j} = -s_{ij} + \log \sum_{k \neq i} \exp(s_{ik}).
$$

Differentiate with respect to $\mathbf{z}_i$, using $\partial s_{ik} / \partial \mathbf{z}_i = \mathbf{z}_k / \tau$:

$$
\frac{\partial \ell_{i,j}}{\partial \mathbf{z}_i}
= -\frac{\mathbf{z}_j}{\tau} + \sum_{k \neq i} p_{ik}\, \frac{\mathbf{z}_k}{\tau}
= \frac{1}{\tau}\Big[ \underbrace{(p_{ij} - 1)\, \mathbf{z}_j}_{\text{pull toward the positive}} + \underbrace{\sum_{k \neq i, j} p_{ik}\, \mathbf{z}_k}_{\text{push away from negatives}} \Big].
$$


A gradient-descent step moves $\mathbf{z}_i$ **toward** $\mathbf{z}_j$ with weight $(1 - p_{ij})$ and **away** from each negative $\mathbf{z}_k$ with weight $p_{ik}$. This is the same softmax-minus-one-hot gradient as the cross-entropy of lecture 3; the "classes" are the other embeddings.


A numerical check of the derivation. Take $2N$ unit vectors, make
$\mathbf{z}_j$ a near copy of $\mathbf{z}_i$ (the positive) and $\mathbf{z}_1$
a hard negative, evaluate the analytic gradient, and compare it with central
finite differences of $\ell_{i,j}$. As on the slide, $\mathbf{z}_i$ is varied
freely and not renormalized. The last lines show where the push goes: the hard
negative gets most of the weight $p_{ik}$, and lowering $\tau$ concentrates it
further.

In [ ]:
import numpy as np

N, d, tau = 4, 16, 0.1     # positive pairs (2N examples), embedding width, temperature
i, j = 0, 4                # the anchor and its positive
rng = np.random.default_rng(0)

def unit(x):
    return x / np.linalg.norm(x, axis=-1, keepdims=True)

Z = unit(rng.normal(size=(2 * N, d)))
Z[j] = unit(Z[i] + 0.3 * rng.normal(size=d) / np.sqrt(d))   # the other view of the same image
Z[1] = unit(Z[i] + 0.8 * rng.normal(size=d) / np.sqrt(d))   # a hard negative

def nt_xent(z_i, Z, i, j, tau):
    s = Z @ z_i / tau
    s[i] = -np.inf                             # the anchor is not its own candidate
    return -s[j] + np.log(np.exp(s).sum())

def nt_xent_grad(Z, i, j, tau):
    s = Z @ Z[i] / tau
    s[i] = -np.inf
    p = np.exp(s - s.max())
    p /= p.sum()                               # p_ik over k != i; p_ii = 0
    pull = (p[j] - 1) * Z[j]
    push = sum(p[k] * Z[k] for k in range(len(Z)) if k not in (i, j))
    return (pull + push) / tau, p

g, p = nt_xent_grad(Z, i, j, tau)
h = 1e-6
g_fd = np.array([(nt_xent(Z[i] + h * e, Z, i, j, tau) - nt_xent(Z[i] - h * e, Z, i, j, tau)) / (2 * h)
                 for e in np.eye(d)])
print(f"loss = {nt_xent(Z[i], Z, i, j, tau):.4f}")
print(f"max |analytic - finite difference| = {np.abs(g - g_fd).max():.2e} "
      f"(gradient norm {np.linalg.norm(g):.2f})")

print(f"\ncosine to z_i: {np.round(Z @ Z[i], 2)}")
for t in [0.5, 0.1, 0.05]:
    _, p_t = nt_xent_grad(Z, i, j, t)
    print(f"tau = {t:<4}  p_ik = {np.round(p_t, 3)}  (positive {p_t[j]:.3f})")

## What the gradient tells us

- **Hard negatives dominate.** The push from negative $k$ is proportional to $p_{ik}$, so a negative that already looks like $\mathbf{z}_i$ is pushed hardest; distant negatives contribute almost nothing. Random negatives would waste the batch
- **Temperature sets how hard.** Small $\tau$ sharpens $p$, so the push concentrates on the few hardest negatives; large $\tau$ spreads it uniformly. $\tau$ too small and the model fixates on near-duplicates, too large and the signal is weak: hence $\tau \approx 0.1$ and CLIP's learned $\exp(t)$
- **The gradient vanishes once the positive wins.** When $p_{ij} \to 1$, both terms go to 0: a solved pair stops contributing, which is why more, and harder, pairs (big batches) keep helping
- **Alignment and uniformity.** [Wang and Isola, 2020](https://arxiv.org/abs/2005.10242) show the loss decomposes into a term pulling positives together and a term spreading all embeddings uniformly on the sphere. Uniformity is what prevents **collapse**, the trivial solution where every image maps to the same point


## Augmentations are the supervision

- The only "label" in SimCLR is "these two views came from the same image." What the encoder learns is **invariance to the augmentation family** $\mathcal{T}$, so the choice of $\mathcal{T}$ *is* the choice of what to learn
- SimCLR's ablation (Figure 5 of the paper): random crop plus color distortion is the pair that matters. Crop alone fails because the two crops of one image share a color histogram, and the model solves the task from color statistics
- The projection head $\mathrm{g}[\cdot]$ matters: the loss makes $\mathbf{z}$ invariant to augmentation, which throws away color and position; $\mathbf{h}$, one layer earlier, keeps that information and transfers better
- Results: 76.5% ImageNet top-1 with a linear classifier on frozen features (ResNet-50 4x wide), a 7% relative jump over the prior state of the art; 85.8% top-5 after fine-tuning with only 1% of the labels
- Successors remove the need for explicit negatives: MoCo keeps a queue of past embeddings ([He et al., 2020](https://arxiv.org/abs/1911.05722)); BYOL uses a momentum target network and no negatives at all ([Grill et al., 2020](https://arxiv.org/abs/2006.07733)), which leads to DINO


## Masked image modeling: MAE

![Figure 1 of [He et al., 2022](https://arxiv.org/abs/2111.06377).](figs/16-Vision-Transformers-and-SSL/mae-architecture.png)

[He, Chen, Xie, Li, Dollár and Girshick, "Masked Autoencoders Are Scalable Vision Learners"](https://arxiv.org/abs/2111.06377), CVPR 2022. BERT's objective for images: hide patches, predict them.

## MAE: what makes it work

- **Mask 75% of patches**, not BERT's 15%. Images are redundant: a missing patch can be inpainted from its neighbors, so a low masking ratio makes the task trivial. At 75% the model must know what the object is
- **Asymmetric encoder/decoder.** The encoder (ViT-L or ViT-H) sees only the 25% visible patches; a small decoder (8 blocks, width 512) takes encoded patches plus mask tokens and predicts pixels. Encoder compute drops by 3 to 4x, so pre-training is fast
- **Target: normalized pixels** of each masked patch (per-patch mean and std), mean-squared error on the masked patches only
- After pre-training, the decoder is discarded and the encoder sees full images


Results on ImageNet-1K (no external data):

| | fine-tune | linear probe |
|---|---|---|
| ViT-L, MAE | 84.9 | 73.5 |
| ViT-L, 1 block tuned | 81.0 | |
| ViT-H/14 at 448 px | **87.8** | |

Fine-tuning is excellent; the linear probe is weak, because the pixel task leaves the last layers doing reconstruction, not semantics. Tuning a *single* block jumps 73.5 to 81.0.


## Self-distillation: DINO

![Figure 2 of [Caron et al., 2021](https://arxiv.org/abs/2104.14294).](figs/16-Vision-Transformers-and-SSL/dino-framework.png)


[Caron *et al.*, "Emerging Properties in Self-Supervised Vision Transformers"](https://arxiv.org/abs/2104.14294), ICCV 2021. Knowledge distillation **with no teacher**:

- Student $g_{\theta_s}$ and teacher $g_{\theta_t}$ have the same architecture; the teacher's weights are an exponential moving average of the student's (no gradient)
- Both output a $K$-way softmax; the loss is the cross-entropy $-P_t \log P_s$ of the student against the teacher
- **Multi-crop:** two global crops go through the teacher, those plus several small local crops go through the student, so the student must predict the whole from a part
- Collapse is avoided with only **centering** (subtract a running mean from the teacher logits, so no one dimension dominates) and **sharpening** (a low teacher temperature, so the output is not uniform)


## What DINO features look like

![Self-attention of the `[CLS]` token in the last layer of a DINO-trained ViT-S/8, three heads in color, no segmentation labels used. Figure 4 of [Caron et al., 2021](https://arxiv.org/abs/2104.14294).](figs/16-Vision-Transformers-and-SSL/dino-attention-maps.png)

- The attention maps segment objects **without any supervision**: the structure of the scene emerges from the self-distillation objective
- Features are good *as they are*: a $k$-nearest-neighbor classifier on frozen features gets 77.4% on ImageNet (ViT-B/8), and a linear probe 80.1%, beating the contrastive methods of the time
- This is the property a backbone needs: useful features with no fine-tuning


## DINOv2: curated data and scale

[Oquab *et al.*, "DINOv2: Learning Robust Visual Features without Supervision"](https://arxiv.org/abs/2304.07193), 2023 (Meta).

- **Data curation instead of more data.** LVD-142M: 142M images selected from 1.2B web images by retrieving, for each image in a curated seed set (ImageNet and other benchmarks), its nearest neighbors in embedding space, then de-duplicating. The paper's claim: SSL "can produce such features if trained on enough *curated* data"
- **Objective:** the DINO image-level loss plus the iBOT patch-level loss ([Zhou et al., 2021](https://arxiv.org/abs/2111.07832): masked patches, predicted in the teacher's token space, which is masked modeling without pixels), Sinkhorn-Knopp centering and a KoLeo regularizer that spreads features uniformly
- **Scale:** a ViT-g/14 with 1.1B parameters, then **distilled** into ViT-S, ViT-B and ViT-L students, which are better than training those sizes from scratch
- Frozen DINOv2 features work for classification, depth estimation, semantic segmentation and retrieval, without fine-tuning; the distilled models beat OpenCLIP on most benchmarks


## DINOv3 and why DINO features are the default backbone

[Siméoni *et al.*, "DINOv3"](https://arxiv.org/abs/2508.10104), 2025 (Meta). A 7B-parameter ViT trained on LVD-1689M (1.7B curated images) for a very long schedule.

- **The problem:** at this scale, global features keep improving but **dense (patch-level) features degrade** after a few hundred thousand iterations: patches lose their local consistency and segmentation quality drops
- **Gram anchoring.** Keep an early checkpoint as a "Gram teacher" and penalize the student's patch features for changing their *pairwise similarity structure*:
  $$\mathcal{L}_{\text{Gram}} = \big\| \mathbf{X}_S \mathbf{X}_S^\top - \mathbf{X}_G \mathbf{X}_G^\top \big\|_F^2,$$
  with $\mathbf{X}_S, \mathbf{X}_G$ the L2-normalized patch features of student and Gram teacher. Only the $N \times N$ Gram matrix is anchored, so individual features stay free to improve while the local structure is kept
- Result: one frozen backbone leads on both global tasks (about 88% ImageNet linear probe) and dense tasks (55.9 mIoU ADE20k with a frozen backbone and linear head). Distilled ViT-S to ViT-H+ and ConvNeXt variants for deployment
- **Why DINO is the default backbone:** features that are strong *frozen*, at every patch, on every domain, with no text needed. SAM, depth models, robotics policies and many VLM experiments start from DINOv2/v3 weights


## JEPA: predict in representation space

![Joint-embedding predictive architecture, from Figure 2 of [Assran et al., 2023](https://arxiv.org/abs/2301.08243).](figs/16-Vision-Transformers-and-SSL/jepa-schematic.png)


![I-JEPA, Figure 3 of [Assran et al., 2023](https://arxiv.org/abs/2301.08243).](figs/16-Vision-Transformers-and-SSL/ijepa-architecture.png)


[Assran *et al.*, "Self-Supervised Learning from Images with a Joint-Embedding Predictive Architecture"](https://arxiv.org/abs/2301.08243), CVPR 2023.

- Like MAE, predict hidden blocks from a context block. **Unlike MAE, predict their *representations*, not their pixels**
- Target representations come from a target encoder, an EMA of the context encoder (as in DINO), applied to the full image; the loss is $L_2$ in feature space
- Why: pixels force the model to spend capacity on unpredictable detail (texture, exact color). Predicting an abstract representation lets it ignore what cannot be predicted and keep what can: objects, layout, pose
- No augmentations, no negatives, no pixel decoder: ViT-H/14 reaches 79.3% linear probe, and ViT-H/16 at 448 px reaches 81.1%, in under 1,200 GPU-hours, far less than iBOT or MAE at similar accuracy


## V-JEPA 2: the same idea on video

[Assran *et al.*, "V-JEPA 2: Self-Supervised Video Models Enable Understanding, Prediction and Planning"](https://arxiv.org/abs/2506.09985), 2025 (Meta).

- A ViT-g encoder (about 1B parameters) pre-trained on VideoMix22M: 22M videos and images, over 1M hours, by predicting the representations of masked spatio-temporal regions from the visible ones
- Scaling recipe: more data (2M to 22M videos), a bigger encoder (300M to 1B), a longer schedule, and progressive resolution
- Results: 77.3% top-1 on Something-Something v2 (motion understanding) and 39.7 recall-at-5 on Epic-Kitchens-100 action anticipation, frozen
- **V-JEPA 2-AC:** freeze the encoder, train a 300M action-conditioned predictor on 62 hours of unlabeled robot video, then plan by choosing actions whose predicted representation is closest to the goal image's. Zero-shot pick-and-place on a real arm
- This is the "latent prediction" side of the world-model debate in lecture 19: predict in representation space (JEPA) versus generate pixels (video diffusion)


## Comparing the four objectives

![What each self-supervised objective asks of the encoder, and what it rewards.](figs/16-Vision-Transformers-and-SSL/ssl-objectives-comparison.png)

## The four objectives at a glance

| | Contrastive | Masked (MAE) | Self-distillation (DINO) | JEPA |
|---|---|---|---|---|
| Supervision | augmentation pairs (or captions) | hidden pixels | EMA teacher on another crop | EMA teacher's features |
| Needs negatives | yes (big batch) | no | no | no |
| Collapse guard | negatives / uniformity | reconstruction target | centering + sharpening | EMA target + predictor |
| Best at | retrieval, zero-shot (CLIP) | fine-tuning | frozen features, dense tasks | compute efficiency, video |

## How to evaluate a representation

- **Linear probe:** freeze the encoder, fit a linear classifier on its features. Measures linear separability, the property a frozen backbone needs. Cheap, but it misses non-linear structure
- **$k$-NN:** no training at all; classify by the nearest stored features. DINO's favorite, and the strictest test of "features good as they are"
- **Fine-tuning:** update all weights on the downstream task. Measures how good an *initialization* the pre-training is. MAE's favorite, and the two protocols rank methods differently: MAE ViT-L is 84.9 fine-tuned but 73.5 on a linear probe, while the contrastive MoCo v3 probes higher and fine-tunes lower ([He et al., 2022](https://arxiv.org/abs/2111.06377), Section 4)
- **Dense tasks:** segmentation, depth, correspondence, with a frozen backbone and a light head. This is where DINOv2/v3 separate from CLIP-style encoders, whose features are global by construction
- A number without its protocol is meaningless: "ImageNet 80%" means nothing until you say linear, $k$-NN, or fine-tuned, and whether external data was used


# Foundation models for vision

With a backbone that produces good features frozen, the next step is a model
that solves a whole task family from prompts, trained on data the model itself
helped to label. This section covers the two that every practitioner now
reaches for, segmentation (SAM) and open-vocabulary detection (Grounding
DINO), and the recurring decision of whether to fine-tune a backbone or freeze
it.


## The pattern: backbone plus promptable head

- A **vision foundation model** = a pre-trained backbone (ViT, usually MAE- or DINO-initialized) + a task interface general enough to cover a whole family of tasks without retraining
- The interface is a **prompt**: a point, a box, a mask, a class name or a sentence, instead of a fixed label set
- The language-model analogy is deliberate: pretrain once at great cost, then adapt by prompting, by a small head, or by fine-tuning
- Training such a model needs annotation at a scale no human team can produce, so each one is paired with a **data engine**: the model labels, humans correct, the model is retrained, repeat


## SAM: promptable segmentation

![Figure 1 of [Kirillov et al., 2023](https://arxiv.org/abs/2304.02643).](figs/16-Vision-Transformers-and-SSL/sam-overview.png)

[Kirillov *et al.*, "Segment Anything"](https://arxiv.org/abs/2304.02643), ICCV 2023 (Meta).

- **Task:** given any prompt (points, box, mask), return a valid mask, even when the prompt is ambiguous (SAM returns three candidate masks: whole, part, sub-part)
- **Model:** an MAE-pretrained ViT-H image encoder, run once per image; a prompt encoder; a lightweight mask decoder that runs in milliseconds in a browser, so interaction is real-time
- **Data:** SA-1B, 1.1B masks on 11M licensed images, 99.1% of them generated automatically by SAM itself


## SAM's data engine, and SAM 2

**Three stages of the data engine**

1. *Assisted-manual:* annotators click points, SAM (trained on public datasets) proposes masks, they correct; SAM is retrained on the new masks, 6 times
2. *Semi-automatic:* SAM pre-fills the confident masks, annotators label what it missed, for diversity
3. *Fully automatic:* prompt SAM with a $32 \times 32$ grid of points per image, keep the stable, confident masks. This stage produced 99.1% of the masks

The model makes the dataset that makes the model.


**SAM 2** ([Ravi et al., 2024](https://arxiv.org/abs/2408.00714)): the same promptable task on **video**

- A streaming **memory**: a FIFO bank of recent frames' features and of prompted frames, attended by the current frame, plus object pointers; a mask propagates through the video from one click
- Hiera image encoder, MAE-pretrained, so one model handles images (6x faster than SAM) and video (3x fewer interactions than prior methods)
- Its own data engine: SA-V, 50.9K videos with 642.6K masklets; annotation time per frame fell 8.4x from stage 1 to stage 3


**SAM 3** (Meta, Nov 2025, [arXiv:2511.16719](https://arxiv.org/abs/2511.16719)) adds concept prompts (text and exemplars).

## Grounding DINO: open-vocabulary detection

[Liu *et al.*, "Grounding DINO: Marrying DINO with Grounded Pre-Training for Open-Set Object Detection"](https://arxiv.org/abs/2303.05499), 2023.

- **Name clash:** this DINO is the DETR-style detector of [Zhang et al., 2022](https://arxiv.org/abs/2203.03605) ("DETR with Improved deNoising anchOr boxes"), not the self-distillation DINO. Two unrelated Meta-adjacent acronyms; check which one a paper means
- A transformer detector takes the image **and a text prompt** ("chair . person . dog ."); language and image features are fused in three places (feature enhancer, language-guided query selection, cross-modality decoder) and each predicted box is scored against the phrase it grounds
- Trained on detection plus phrase-grounding data, it detects **any category named in the prompt**: 52.5 AP on COCO with no COCO training images, 26.1 mean AP on the 35 ODinW datasets zero-shot
- **Grounded SAM** ([Ren et al., 2024](https://arxiv.org/abs/2401.14159)): text to Grounding DINO to boxes to SAM to masks. The standard open-vocabulary segmentation pipeline, built from two frozen foundation models and no training


## Fine-tune or freeze?

| Situation | Use | Why |
|---|---|---|
| Few labels (< 10K), domain like the web | **frozen DINOv2/v3 + linear or small head** | the features are already linearly useful; nothing to overfit |
| Need text (retrieval, zero-shot classes) | **frozen CLIP / SigLIP** | the only backbones with a shared text space |
| Segmentation or detection with named classes | **Grounded SAM, zero-shot** | no training; then fine-tune the heads if accuracy is short |
| Many labels, domain far from the web (medical, satellite, microscopy) | **fine-tune an MAE- or DINO-initialized ViT** | initialization transfers; the last layers need to change |
| Dense task, strict latency budget | **distilled ViT-S/B or ConvNeXt from DINOv3** | same features, 10 to 100x cheaper |

Fine-tuning an entire ViT-L on a small dataset usually *hurts*: it erases the pre-trained features before the task has enough signal to replace them. Try the frozen backbone first, then LoRA (lecture 13), then partial fine-tuning of the last blocks (MAE's Figure 9: one block gave most of the gain).


# Vision-language models

The last section connects this lecture to the language-model half of the
course. Every current multimodal LLM feeds the output of a vision encoder,
usually a CLIP-family or SigLIP ViT, into a language model through a small
projector, and the only real design question left is how many tokens an image
is allowed to cost. That calculation is the lecture's last quizzable item.


## The pattern: encoder + projector + LLM

![LLaVA, Figure 1 of [Liu et al., 2023](https://arxiv.org/abs/2304.08485).](figs/16-Vision-Transformers-and-SSL/llava-architecture.png)

- [Liu *et al.*, "Visual Instruction Tuning"](https://arxiv.org/abs/2304.08485) (LLaVA), NeurIPS 2023: the frozen **CLIP ViT-L/14** gives one feature per patch, $\mathbf{Z}_v$; a trainable projection $\mathbf{W}$ maps each patch feature into the LLM's token-embedding space; the LLM (Vicuna) then sees **image tokens followed by text tokens** and generates as usual
- Nothing in the LLM changes. An image is a run of a few hundred "words" in a foreign embedding, and the transformer's attention does the cross-modal work
- [LLaVA-1.5](https://arxiv.org/abs/2310.03744): a two-layer MLP projector, CLIP at 336 px ($24 \times 24 = 576$ tokens per image), and better data. Still the template for most open VLMs


## Training stages

1. **Projector alignment.** Freeze encoder and LLM, train only $\mathbf{W}$ on image-caption pairs (LLaVA: 595K pairs from CC3M, a few GPU-hours). The projector learns to speak the LLM's embedding language
2. **Visual instruction tuning.** Unfreeze the LLM (and often the projector), train on (image, instruction, response) triples: LLaVA's 158K GPT-4-generated conversations, descriptions and reasoning questions. This is SFT (lecture 13) with images in the prompt
3. **Current recipe (Qwen2.5-VL, Qwen3-VL, Gemma 3):** a multimodal *pre-training* stage on **interleaved image-text documents**, OCR, documents, charts, grounding data and video, at the trillion-token scale (Qwen2.5-VL: about 4T tokens over three stages), *before* instruction tuning; then the same SFT and RL stages as a text model (lectures 13 to 14)


Whether to unfreeze the vision encoder is a live choice: Gemma 3 keeps its SigLIP encoder **frozen**; Qwen2.5-VL trains its ViT from scratch and Qwen3-VL unfreezes everything in the joint stage. Unfreezing helps OCR and fine detail; freezing preserves the encoder's robustness and saves compute.


## Current open VLMs and their vision encoders

| Model (2025) | Vision encoder | Patch, merge | Tokens per image | LLM sizes |
|---|---|---|---|---|
| [Qwen2.5-VL](https://arxiv.org/abs/2502.13923) | ViT trained from scratch, window attention, 2-D RoPE | 14 px, $2 \times 2$ merge: **28 px per token** | dynamic, $(H/28)(W/28)$, 4 to 16,384 | 3B, 7B, 72B |
| [Qwen3-VL](https://arxiv.org/abs/2511.21631) | **SigLIP 2** (SO-400M), unfrozen | 16 px, $2 \times 2$ merge: **32 px per token** | dynamic, $(H/32)(W/32)$ | 2B to 32B dense; 30B-A3B, 235B-A22B MoE |
| [Gemma 3](https://arxiv.org/abs/2503.19786) | **SigLIP** (400M), frozen | 14 px at fixed 896 px, pooled | **256**, fixed; Pan & Scan crops for large images | 4B, 12B, 27B |
| [LLaVA-1.5](https://arxiv.org/abs/2310.03744) (2023) | CLIP ViT-L/14 at 336 px | 14 px | **576**, fixed | 7B, 13B |

- SigLIP ([Zhai et al., 2023](https://arxiv.org/abs/2303.15343)) is CLIP with a per-pair sigmoid loss instead of the batch softmax; it trains well at smaller batch sizes and is the encoder in most 2025 VLMs. SigLIP 2: [Tschannen et al., 2025](https://arxiv.org/abs/2502.14786)
- Qwen3-VL's **DeepStack** also injects features from three intermediate ViT layers into the first LLM layers, so the LLM sees low-level detail without extra tokens
- Keep system names on this one slide; the pattern on the previous two is what lasts

## Dynamic resolution and the token budget

In [ ]:
#| echo: false
#| fig-cap: "Left: visual tokens per square image as a function of side length, for fixed-token and dynamic-resolution designs. Right: the KV cache those tokens occupy in two open models (bf16). Token formulas from the cited papers; KV sizes from the public model configs."
#| fig-alt: "Two line plots against image side length from 224 to 2048 pixels. Left: visual tokens per image. Qwen2.5-VL grows as (side/28) squared to over 5000 tokens at 2048 px, Qwen3-VL as (side/32) squared to about 4100; LLaVA-1.5 is flat at 576 and Gemma 3 flat at 256; annotations mark 448 px: 256 tokens and 1024 px: 1,369 tokens for Qwen2.5-VL. Right: KV cache in MiB; Qwen2.5-VL-7B at 56 KiB per token reaches about 290 MiB at 2048 px and Qwen3-VL-8B at 144 KiB per token about 580 MiB; an annotation reads 1024 px: 1,369 tokens x 56 KiB = 75 MiB."
import numpy as np
import matplotlib.pyplot as plt

side = np.arange(224, 2049, 16)
qwen25 = np.round(side / 28) ** 2           # patch 14, 2x2 merge: 28 px per token
qwen3 = np.round(side / 32) ** 2            # patch 16, 2x2 merge: 32 px per token
kv_qwen25 = 2 * 28 * 4 * 128 * 2            # bytes per token: 2 x layers x KV heads x d_head x bf16
kv_qwen3 = 2 * 36 * 8 * 128 * 2
BLUE, AQUA, ORANGE, VIOLET = "#2a78d6", "#1baf7a", "#eb6834", "#4a3aa7"

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.8), dpi=76)
ax1.plot(side, qwen25, color=BLUE, lw=2, label="Qwen2.5-VL: (side/28)$^2$, dynamic")
ax1.plot(side, qwen3, color=AQUA, lw=2, label="Qwen3-VL: (side/32)$^2$, dynamic")
ax1.plot(side, np.full(side.shape, 576), color=ORANGE, lw=2, ls="--", label="LLaVA-1.5: 576, fixed (336 px in)")
ax1.plot(side, np.full(side.shape, 256), color=VIOLET, lw=2, ls="--", label="Gemma 3: 256, fixed (896 px in)")
for s in (448, 1024):
    n = round(s / 28) ** 2
    ax1.plot([s], [n], "o", color=BLUE, ms=6)
    ax1.annotate(f"{s} px: {n:,} tokens", (s, n), textcoords="offset points",
                 xytext=(8, -14), fontsize=9, color=BLUE)
ax1.set_ylabel("visual tokens per image")
ax1.set_title("How many tokens does one image cost?", fontsize=11)

ax2.plot(side, qwen25 * kv_qwen25 / 2**20, color=BLUE, lw=2,
         label=f"Qwen2.5-VL-7B: {kv_qwen25 // 1024} KiB per token")
ax2.plot(side, qwen3 * kv_qwen3 / 2**20, color=AQUA, lw=2,
         label=f"Qwen3-VL-8B: {kv_qwen3 // 1024} KiB per token")
n = round(1024 / 28) ** 2
mib = n * kv_qwen25 / 2**20
ax2.annotate(f"1024 px: {n:,} tokens x {kv_qwen25 // 1024} KiB = {mib:.0f} MiB",
             (1024, mib), textcoords="offset points", xytext=(8, -16), fontsize=9, color=BLUE)
ax2.set_ylabel("KV cache for the image (MiB, bf16)")
ax2.set_title("What the KV cache pays for it (lecture 15)", fontsize=11)
for ax in (ax1, ax2):
    ax.set_xlabel("image side length (pixels, square image)")
    ax.legend(fontsize=9, loc="upper left", frameon=False)
    ax.grid(color="#dcdcd8", lw=0.6)
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()

- Fixed designs (LLaVA, Gemma 3) resize every image to one size: predictable cost, but small text and fine detail are lost; Gemma 3's Pan & Scan tiles a large image into 896-px crops at 256 tokens each
- Dynamic designs (Qwen) keep the native resolution: tokens scale with **area**, so a 2x larger side costs 4x the tokens. Processors cap the area (Qwen2.5-VL: 12.8M pixels, 16,384 tokens) and round each side to the token grid


## The quizzable calculation: tokens and KV cache

**Tokens per image.** Patch size $P$, spatial merge $m \times m$, image $H \times W$ (each side rounded to a multiple of $mP$):

$$
N_{\text{vis}} = \frac{H}{mP} \cdot \frac{W}{mP}.
$$

Qwen2.5-VL ($P = 14$, $m = 2$, so 28 px per token), $1024 \times 1024$ image: sides round to $1036 = 37 \times 28$, so $N_{\text{vis}} = 37 \times 37 = 1{,}369$ tokens. Qwen3-VL (32 px per token): $32 \times 32 = 1{,}024$. LLaVA-1.5: always 576. Gemma 3: always 256.


**KV cache per token** (lecture 15): $2 \times L \times n_{\text{kv}} \times d_{\text{head}} \times \text{bytes}$. Qwen2.5-VL-7B: 28 layers, 4 KV heads (GQA, lecture 11), $d_{\text{head}} = 128$, bf16:

$$
2 \times 28 \times 4 \times 128 \times 2 = 57{,}344 \text{ bytes} = 56 \text{ KiB per token}.
$$


So the 1024-px image occupies $1{,}369 \times 56 \text{ KiB} \approx 75 \text{ MiB}$ of KV cache for as long as it is in context; the 16,384-token maximum costs 0.9 GiB; a batch of 32 such requests costs 2.3 GiB *before* any text. Qwen3-VL-8B (36 layers, 8 KV heads) pays 144 KiB per token. This is why VLM serving is memory-bound and why token budgets, not model size, decide throughput.


The same calculation in code, using the processor's rounding (each side to the
nearest multiple of $mP$). Change the image size or swap in another model's
patch, merge and KV-cache shape.

In [ ]:
P, m = 14, 2                  # Qwen2.5-VL: patch size, spatial merge
L, n_kv, d_head = 28, 4, 128  # Qwen2.5-VL-7B: layers, KV heads, head dimension
bytes_per = 2                 # bf16
H, W = 1024, 1024             # image size in pixels

KiB, MiB, GiB = 2**10, 2**20, 2**30

def visual_tokens(H, W, P=P, m=m):
    unit = m * P
    return round(H / unit) * round(W / unit)

kv_per_token = 2 * L * n_kv * d_head * bytes_per
n_vis = visual_tokens(H, W)
image_kv = n_vis * kv_per_token
print(f"{H}x{W}: sides round to {round(H / (m * P)) * m * P}x{round(W / (m * P)) * m * P}, "
      f"N_vis = {round(H / (m * P))} x {round(W / (m * P))} = {n_vis:,} tokens")
print(f"KV cache per token: {kv_per_token:,} B = {kv_per_token / KiB:.0f} KiB")
print(f"the image's KV cache: {image_kv / MiB:.1f} MiB; "
      f"a batch of 32 such images: {32 * image_kv / GiB:.2f} GiB")
print(f"16,384-token maximum: {16384 * kv_per_token / GiB:.3f} GiB")
print(f"Qwen3-VL (32 px per token) at {H}x{W}: {visual_tokens(H, W, P=16):,} tokens, "
      f"{2 * 36 * 8 * 128 * 2 / KiB:.0f} KiB per token")

fps_pairs = 1                 # 2 fps, two frames per temporal patch
for name, (h, w) in {"448x448": (448, 448), "720p": (720, 1280)}.items():
    print(f"one minute of {name} video at 2 fps: {60 * fps_pairs * visual_tokens(h, w):,} tokens")

## Video tokens

- A video is a stack of frames, so the arithmetic multiplies by time. Qwen2.5-VL samples frames at a dynamic rate and groups **two consecutive frames into one temporal patch** ($2 \times 14 \times 14$), so with 2 fps sampling, one second costs $N_{\text{vis}}$ tokens, the same as one image at that resolution
- A one-minute clip at $448 \times 448$ and 2 fps: $60 \times 16 \times 16 = 15{,}360$ tokens, before any text. At 720p it would be about 70K. Processors therefore cap total pixels per video and downsample frames, and Qwen2.5-VL caps at 768 frames
- Position: Qwen2.5-VL's multimodal RoPE (lecture 10) gives each token a (time, height, width) triple, with the time index aligned to **absolute seconds**, so the model can localize events; Qwen3-VL instead inserts **text timestamps** (`<3.0 seconds>`) between frame groups and interleaves the RoPE frequencies across the three axes
- Lecture 19 looks at the other side of video: generating it, and world models (V-JEPA 2) that predict it in representation space


## Limits and open questions

- **SSL on images has no "next token."** Each objective encodes a guess about what matters (invariance, pixels, consistency, predictability); none is as neutral as language modeling, and the best choice is still task-dependent
- **Text supervision is biased.** CLIP-family encoders know what the web captions, not what a radiologist sees; DINO-family encoders know structure but not names. Current VLMs lean on CLIP/SigLIP, and that choice limits fine-grained perception
- **Tokens are the bottleneck.** Resolution, video length and multi-image prompts all compete for a KV-cache budget; token compression (merging, pooling, DeepStack-style injection) is where VLM architecture research is in 2026
- **Dense and 3-D understanding** (depth, correspondence, physical plausibility) is where frozen SSL backbones beat VLMs, and where JEPA-style world models are heading (lecture 19)


# Summary

## Summary

- **ViT**: cut the image into $P \times P$ patches, $N = HW/P^2$ tokens, standard pre-norm encoder, `[CLS]` head; it beats CNNs only with 14M to 300M images, at a fraction of the compute
- **CLIP**: a ViT and a text transformer trained so matching (image, caption) pairs have high cosine similarity; symmetric cross-entropy over the batch, learned inverse temperature $\exp(t)$; zero-shot classification by encoding class names
- **Contrastive SSL (SimCLR)**: NT-Xent pulls two augmentations of an image together and pushes other images away; the gradient weights each negative by its softmax probability, so hard negatives and temperature matter; augmentations are the supervision
- **Masked modeling (MAE)**: hide 75% of patches, encode the rest, reconstruct pixels with a small decoder; fine-tunes superbly, probes poorly
- **Self-distillation (DINO, DINOv2, DINOv3)**: a student matches an EMA teacher across crops, with centering and sharpening against collapse; curated data and Gram anchoring scale it to 7B; its frozen features are the default backbone
- **JEPA (I-JEPA, V-JEPA 2)**: predict the *representations* of hidden regions, not pixels; efficient, and the basis of latent world models (lecture 19)
- **Evaluation**: linear probe, $k$-NN and fine-tuning rank methods differently; always state the protocol
- **Foundation models**: SAM (promptable masks, a data engine that produced 1.1B masks), SAM 2 (video with memory), Grounding DINO (detect what you name); freeze first, fine-tune last
- **VLMs**: vision encoder + projector + LLM; image tokens cost $(H/mP)(W/mP)$ each and $2 L n_{\text{kv}} d_{\text{head}} \cdot \text{bytes}$ of KV cache; a 1024-px image is about 1,400 tokens and 75 MiB in Qwen2.5-VL-7B